<a href="https://colab.research.google.com/github/aminaalavi/AI-Projects/blob/main/Matter_Precedent_Risk_Agents/Matter_Precedent_Risk_Agents_no_SL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit aisuite tavily-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 44.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.68.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


In [ ]:
import sys
import os
from google.colab import userdata
import json
from typing import List, Dict, Any

import aisuite
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tavily import TavilyClient
import random


In [ ]:
MODEL_NAME = "openai:o4-mini"
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

# ========= CLIENTS =========



def get_tavily_client():
    if not os.getenv("TAVILY_API_KEY"):
        raise RuntimeError("TAVILY_API_KEY is not set.")
    return TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def get_client():
    """
    Create an aisuite client.
    Requires OPENAI_API_KEY to be set in your environment.
    """
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is not set.")
    return aisuite.Client()


def llm_complete(system_prompt: str, user_prompt: str) -> str:
    """
    Small helper to call the model through aisuite.
    """
    client = get_client()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content.strip()


MODEL_NAME = "openai:gpt-4o-mini"


In [ ]:
# =========================================================
# SYNTHETIC INTERNAL DATA
# =========================================================

def generate_core_past_matters() -> List[Dict[str, Any]]:
    return [
        {
            "id": "M001",
            "practice_area": "Antitrust",
            "industry": "Technology",
            "client_type": "Public Company",
            "matter_summary": (
                "Technology platform company faced antitrust scrutiny over alleged exclusionary conduct, "
                "bundling practices, and market power in a core software ecosystem."
            ),
            "key_issues": [
                "market power",
                "bundling",
                "platform conduct",
                "behavioral remedies",
            ],
            "approach_taken": (
                "Defense focused on consumer benefit narrative, efficiency justification, "
                "and narrowing the relevant market definition."
            ),
            "outcome": "Investigation resolved through negotiated behavioral commitments.",
            "lessons_learned": (
                "Past precedent was helpful, but regulator posture had shifted more aggressively than expected."
            ),
        },
        {
            "id": "M002",
            "practice_area": "M&A",
            "industry": "Healthcare",
            "client_type": "Private Equity",
            "matter_summary": (
                "Private equity sponsor pursued acquisition of healthcare services company with significant "
                "regulatory and concentration concerns."
            ),
            "key_issues": [
                "deal timing",
                "regulatory review",
                "financing complexity",
                "market concentration",
            ],
            "approach_taken": (
                "Cross-practice coordination between M&A, antitrust, and finance teams; mitigation planning began early."
            ),
            "outcome": "Transaction approved after timeline extension and negotiated remedies.",
            "lessons_learned": (
                "The earlier the team modeled regulator objections, the stronger the negotiation position."
            ),
        },
        {
            "id": "M003",
            "practice_area": "White Collar / Regulatory",
            "industry": "Financial Services",
            "client_type": "Public Company",
            "matter_summary": (
                "Financial institution faced investigation tied to disclosure controls, compliance gaps, "
                "and executive-level oversight concerns."
            ),
            "key_issues": [
                "disclosure",
                "controls",
                "oversight",
                "investigation response",
            ],
            "approach_taken": (
                "Internal fact development, regulator engagement strategy, and remediation narrative built in parallel."
            ),
            "outcome": "Matter resolved with penalties lower than initial exposure estimates.",
            "lessons_learned": (
                "Consistency between factual record and remediation story was critical."
            ),
        },
        {
            "id": "M004",
            "practice_area": "Litigation",
            "industry": "Consumer Products",
            "client_type": "Public Company",
            "matter_summary": (
                "Company faced commercial litigation and follow-on reputational risk after allegations of deceptive practices."
            ),
            "key_issues": [
                "consumer harm theory",
                "damages framing",
                "reputation",
                "settlement posture",
            ],
            "approach_taken": (
                "Team split strategy across courtroom posture and parallel business messaging."
            ),
            "outcome": "Case settled before trial under confidential terms.",
            "lessons_learned": (
                "Reputational context mattered almost as much as legal merits."
            ),
        },
        {
            "id": "M005",
            "practice_area": "Antitrust",
            "industry": "Healthcare",
            "client_type": "Strategic Buyer",
            "matter_summary": (
                "Acquirer faced antitrust questions over a vertical integration involving healthcare data and services."
            ),
            "key_issues": [
                "vertical integration",
                "foreclosure theory",
                "data advantage",
                "regulatory challenge",
            ],
            "approach_taken": (
                "Emphasized pro-competitive benefits, interoperability, and lack of foreclosure incentives."
            ),
            "outcome": "Review was prolonged; transaction structure was modified.",
            "lessons_learned": (
                "Older transaction precedent did not cleanly map onto newer regulator concerns about data concentration."
            ),
        },
        {
            "id": "M006",
            "practice_area": "Activism Defense",
            "industry": "Industrials",
            "client_type": "Public Company",
            "matter_summary": (
                "Public company confronted activist pressure tied to governance, capital allocation, and strategic alternatives."
            ),
            "key_issues": [
                "board governance",
                "activist pressure",
                "public narrative",
                "shareholder communication",
            ],
            "approach_taken": (
                "Coordinated activism defense, governance review, and investor messaging."
            ),
            "outcome": "Campaign de-escalated after negotiated governance changes.",
            "lessons_learned": (
                "Speed and internal alignment were more important than legal complexity."
            ),
        },
        {
            "id": "M007",
            "practice_area": "Finance",
            "industry": "Technology",
            "client_type": "Private Equity",
            "matter_summary": (
                "Sponsor-backed financing required restructuring due to market volatility and covenant stress."
            ),
            "key_issues": [
                "liquidity",
                "covenants",
                "market volatility",
                "refinancing risk",
            ],
            "approach_taken": (
                "Debt restructuring workstream coordinated with sponsors, lenders, and transactional counsel."
            ),
            "outcome": "Refinancing completed with tighter economics.",
            "lessons_learned": (
                "Financial stress can quickly create adjacent restructuring and dispute risk."
            ),
        },
        {
            "id": "M008",
            "practice_area": "M&A",
            "industry": "Technology",
            "client_type": "Strategic Buyer",
            "matter_summary": (
                "Technology acquisition triggered extensive diligence around data practices, IP ownership, and integration risk."
            ),
            "key_issues": [
                "IP diligence",
                "data practices",
                "integration risk",
                "deal protections",
            ],
            "approach_taken": (
                "Built diligence escalation framework and negotiated targeted indemnity protections."
            ),
            "outcome": "Deal signed with enhanced protections and post-closing remediation obligations.",
            "lessons_learned": (
                "Operational diligence themes often become legal risk themes after signing."
            ),
        },
    ]


def generate_large_synthetic_dataset(n: int = 60) -> List[Dict[str, Any]]:
    practice_areas = [
        "Antitrust", "M&A", "Litigation", "White Collar / Regulatory",
        "Finance", "Activism Defense"
    ]
    industries = [
        "Technology", "Healthcare", "Financial Services",
        "Energy", "Consumer Products", "Industrials"
    ]
    client_types = [
        "Public Company", "Private Equity", "Strategic Buyer", "Financial Sponsor"
    ]

    issue_bank = {
        "Antitrust": ["market power", "bundling", "pricing practices", "vertical integration", "data concentration"],
        "M&A": ["deal timing", "regulatory review", "financing complexity", "integration risk", "diligence gaps"],
        "Litigation": ["damages exposure", "reputation", "settlement posture", "factual inconsistency", "procedural risk"],
        "White Collar / Regulatory": ["disclosure", "controls", "oversight", "investigation response", "remediation"],
        "Finance": ["liquidity", "covenants", "market volatility", "refinancing risk", "credit stress"],
        "Activism Defense": ["board governance", "shareholder communication", "capital allocation", "strategic alternatives", "public narrative"],
    }

    approach_templates = [
        "Cross-practice coordination was initiated early to align legal and business positioning.",
        "The team focused on narrowing the dispute frame and strengthening the factual narrative.",
        "Mitigation planning began early, with legal and commercial workstreams coordinated in parallel.",
        "The strategy emphasized consistency between regulator-facing and business-facing narratives.",
        "The team paired substantive defense arguments with contingency planning around remedies or restructuring.",
    ]

    outcome_templates = [
        "Matter resolved through negotiated settlement.",
        "Review was prolonged and ultimately required structural changes.",
        "Transaction proceeded after additional diligence and revised protections.",
        "Investigation closed with lower-than-expected penalties.",
        "Dispute de-escalated after coordinated legal and business response.",
    ]

    lesson_templates = [
        "Older precedent was useful, but only after adjusting for changed regulator posture.",
        "Cross-practice coordination materially improved response quality.",
        "Initial precedent matches overstated similarity and had to be narrowed.",
        "Commercial context was as important as legal theory.",
        "The strongest reusable lesson was process discipline rather than substantive precedent alone.",
    ]

    rows = []
    for i in range(1, n + 1):
        practice = random.choice(practice_areas)
        industry = random.choice(industries)
        client_type = random.choice(client_types)
        issues = random.sample(issue_bank[practice], k=3)

        rows.append({
            "id": f"S{i:03d}",
            "practice_area": practice,
            "industry": industry,
            "client_type": client_type,
            "matter_summary": (
                f"{client_type} in {industry} faced a {practice.lower()}-related matter "
                f"involving {issues[0]}, {issues[1]}, and {issues[2]}."
            ),
            "key_issues": issues,
            "approach_taken": random.choice(approach_templates),
            "outcome": random.choice(outcome_templates),
            "lessons_learned": random.choice(lesson_templates),
        })

    return rows


# =========================================================
# RETRIEVAL
# =========================================================

def matter_to_text(m: Dict[str, Any]) -> str:
    return " | ".join([
        m.get("practice_area", ""),
        m.get("industry", ""),
        m.get("client_type", ""),
        m.get("matter_summary", ""),
        " ".join(m.get("key_issues", [])),
        m.get("approach_taken", ""),
        m.get("outcome", ""),
        m.get("lessons_learned", ""),
    ])


def retrieve_similar_matters(
    new_matter: Dict[str, Any],
    past_matters: List[Dict[str, Any]],
    top_k: int = 3,
    min_similarity: float = 0.05,
) -> List[Dict[str, Any]]:
    query_text = " | ".join([
        new_matter.get("practice_area", ""),
        new_matter.get("industry", ""),
        new_matter.get("client_type", ""),
        new_matter.get("matter_summary", ""),
        " ".join(new_matter.get("key_issues", [])),
    ])

    corpus = [matter_to_text(m) for m in past_matters]
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(corpus + [query_text])

    sims = cosine_similarity(X[-1], X[:-1]).flatten()
    ranked_idx = sims.argsort()[::-1]

    results = []
    for idx in ranked_idx:
        if sims[idx] < min_similarity:
            continue
        item = dict(past_matters[idx])
        item["similarity_score"] = float(sims[idx])
        results.append(item)
        if len(results) >= top_k:
            break
    return results


def format_internal_precedents(similar_matters: List[Dict[str, Any]]) -> str:
    if not similar_matters:
        return "No strong internal precedents found."

    chunks = []
    for i, m in enumerate(similar_matters, start=1):
        chunks.append(
            f"""[Internal Precedent {i}]
ID: {m["id"]}
Practice Area: {m["practice_area"]}
Industry: {m["industry"]}
Client Type: {m["client_type"]}
Similarity Score: {m["similarity_score"]:.3f}
Matter Summary: {m["matter_summary"]}
Key Issues: {", ".join(m["key_issues"])}
Approach Taken: {m["approach_taken"]}
Outcome: {m["outcome"]}
Lessons Learned: {m["lessons_learned"]}
"""
        )
    return "\n\n".join(chunks)


# =========================================================
# TAVILY
# =========================================================

RELEVANCE_TERMS = {
    "antitrust", "doj", "ftc", "regulator", "regulatory", "bundling",
    "platform", "market power", "monopoly", "competition", "remedies",
    "structural", "behavioral", "enforcement", "lawsuit", "investigation"
}

NOISY_DOMAINS = {
    "itnews.com.au",
    "thenextweb.com",
    "ecologist.org",
}

NOISY_TITLE_PATTERNS = [
    "in pictures",
    "summit",
    "sentiment is waning",
    "children’s social media use",
]


def generate_tavily_queries(new_matter: Dict[str, Any]) -> List[str]:
    """
    Deterministic, high-precision queries while tuning.
    """
    practice = (new_matter.get("practice_area") or "").lower()
    industry = (new_matter.get("industry") or "").lower()

    if practice == "antitrust" and industry == "technology":
        return [
            "DOJ FTC antitrust tech platform bundling recent",
            "recent antitrust enforcement platform control technology 2025 2026",
            "behavioral remedies versus structural remedies antitrust tech platforms recent",
        ]

    issues = [x.lower() for x in new_matter.get("key_issues", [])]
    key1 = issues[0] if len(issues) > 0 else "core issue"
    key2 = issues[1] if len(issues) > 1 else "regulatory posture"

    return [
        f"recent {practice} {industry} {key1} enforcement",
        f"recent {practice} regulator posture {key2}",
        f"{practice} external developments affecting precedent reliability recent",
    ]


def relevance_score(title: str, summary: str, query: str) -> int:
    text = f"{title} {summary} {query}".lower()
    score = 0

    for term in RELEVANCE_TERMS:
        if term in text:
            score += 1

    for boost in ["doj", "ftc", "antitrust", "platform", "bundling", "remedies"]:
        if boost in text:
            score += 2

    return score


def is_noisy_result(title: str, url: str) -> bool:
    title_l = (title or "").lower()
    url_l = (url or "").lower()

    if any(domain in url_l for domain in NOISY_DOMAINS):
        return True

    if any(pattern in title_l for pattern in NOISY_TITLE_PATTERNS):
        return True

    return False


def tavily_search_packet(queries: List[str], max_results: int = 5) -> Dict[str, Any]:
    client = get_tavily_client()
    items = []

    for q in queries:
        try:
            result = client.search(
                query=q,
                search_depth="advanced",
                topic="news",
                max_results=max_results,
                include_answer=True,
                include_raw_content=False,
            )

            answer = (result.get("answer") or "").strip()
            if answer:
                items.append({
                    "kind": "answer",
                    "query": q,
                    "title": f"Tavily answer for: {q}",
                    "url": "",
                    "summary": answer,
                    "score": relevance_score(f"Tavily answer for: {q}", answer, q),
                })

            for r in result.get("results", []):
                title = (r.get("title") or "").strip()
                url = (r.get("url") or "").strip()
                summary = (r.get("content") or "").strip()

                if is_noisy_result(title, url):
                    continue

                score = relevance_score(title, summary, q)
                if score < 3:
                    continue

                items.append({
                    "kind": "result",
                    "query": q,
                    "title": title or "Untitled",
                    "url": url,
                    "summary": summary[:900],
                    "score": score,
                })

        except Exception as e:
            items.append({
                "kind": "error",
                "query": q,
                "title": "Tavily error",
                "url": "",
                "summary": f"Search failed: {e}",
                "score": 0,
            })

    deduped = []
    seen = set()
    for item in sorted(items, key=lambda x: x.get("score", 0), reverse=True):
        key = (item.get("url") or item.get("title") or "").strip().lower()
        if not key:
            continue
        if key in seen:
            continue
        seen.add(key)
        deduped.append(item)

    return {"queries": queries, "items": deduped}


def format_external_context(packet: Dict[str, Any], max_items: int = 4) -> str:
    usable = [
        x for x in packet.get("items", [])
        if x.get("kind") in {"answer", "result"}
    ][:max_items]

    if not usable:
        return "No strong external public context retrieved."

    chunks = []
    for i, item in enumerate(usable, start=1):
        chunks.append(
            f"""[External Context {i}]
Query: {item.get("query", "")}
Title: {item.get("title", "")}
URL: {item.get("url", "")}
Relevance Score: {item.get("score", 0)}
Summary: {item.get("summary", "")}
"""
        )
    return "\n\n".join(chunks)


def format_external_sources(packet: Dict[str, Any], max_items: int = 8) -> str:
    rows = [
        x for x in packet.get("items", [])
        if x.get("kind") == "result"
    ][:max_items]

    if not rows:
        return "No strong public sources."

    return "\n".join(
        f"- {r['title']} | {r['url']} | score={r.get('score', 0)} | query={r['query']}"
        for r in rows
    )


# =========================================================
# AGENTS
# =========================================================

AGENTS = [
    {
        "name": "Precedent Advocate",
        "system": (
            "You are a senior partner arguing that prior firm matters provide a reliable playbook "
            "for the current situation. You strongly believe in precedent as a guide to strategy."
        ),
        "instructions": (
            "Make a confident argument for why internal precedent applies.\n"
            "- Use exactly ONE internal precedent as your anchor\n"
            "- Name one reusable tactic or lesson\n"
            "- Explain why it fits this matter\n"
            "- Focus on what the team should actively reuse\n"
            "- Avoid discussing external context unless needed for contrast"
        ),
        "turn_format": (
            "Output exactly 3 bullets:\n"
            "1. Anchor precedent\n"
            "2. Why it applies\n"
            "3. Reusable tactic"
        ),
    },
    {
        "name": "Skeptic / Challenger",
        "system": (
            "You are a critical reviewer whose job is to prevent the team from relying on misleading precedent. "
            "You assume most analogies are only partially valid."
        ),
        "instructions": (
            "Challenge the strongest precedent argument.\n"
            "- Identify exactly TWO reasons the analogy weakens or breaks\n"
            "- Focus on mismatch in facts, structure, posture, or risk profile\n"
            "- Do not simply say regulators are tougher now unless tied to a concrete mismatch\n"
            "- Your job is to show where the analogy stops working"
        ),
        "turn_format": (
            "Output exactly 3 bullets:\n"
            "1. Main flaw in analogy\n"
            "2. Secondary flaw\n"
            "3. Risk of relying on it"
        ),
    },
    {
        "name": "External Context Agent",
        "system": (
            "You are responsible for grounding the discussion in real-world developments. "
            "You use external signals to determine whether internal precedent is still valid."
        ),
        "instructions": (
            "Use external public context to test precedent.\n"
            "- Cite exactly ONE concrete external development, case, or pattern\n"
            "- Explain what it changes for this matter\n"
            "- State whether it strengthens, weakens, or creates mixed evidence for reliance on precedent\n"
            "- If the external sources are only weakly relevant, say that explicitly and lower your confidence\n"
            "- If the external signal supports the precedent, say it strengthens it\n"
            "- If the external signal is mixed, say the evidence is mixed\n"
            "- Do not claim the precedent is weakened unless the external context clearly cuts against it\n"
            "- Do not overstate weak evidence"
        ),
        "turn_format": (
            "Output exactly 3 bullets:\n"
            "1. External signal\n"
            "2. What it changes\n"
            "3. Conclusion for precedent reliability"
        ),
    },
    {
        "name": "Blind Spot Detector",
        "system": (
            "You are responsible for identifying what the team is missing. "
            "You assume both precedent and external context may be incomplete."
        ),
        "instructions": (
            "Identify exactly ONE important blind spot.\n"
            "- It must be something not already covered by the other agents\n"
            "- Explain why it matters for decision-making\n"
            "- State what the team should test or clarify next"
        ),
        "turn_format": (
            "Output exactly 3 bullets:\n"
            "1. Blind spot\n"
            "2. Why it matters\n"
            "3. What the team needs to test"
        ),
    },
    {
        "name": "Synthesizer",
        "system": (
            "You are a senior decision-maker responsible for turning conflicting views into a clear, actionable position."
        ),
        "instructions": (
            "Make a call.\n"
            "- State what precedent is still usable\n"
            "- State what the team should NOT assume\n"
            "- Give exactly TWO legal workstream actions\n"
            "- Use law-firm practical language: prepare, pressure-test, develop, model, engage, preserve\n"
            "- Do not summarize the room; decide"
        ),
        "turn_format": (
            "Output exactly 4 bullets:\n"
            "1. What still holds\n"
            "2. What not to assume\n"
            "3. Action 1\n"
            "4. Action 2"
        ),
    },
]


def run_agent_turn(
    agent: Dict[str, str],
    new_matter: Dict[str, Any],
    internal_context: str,
    external_context: str,
    prior_discussion: str,
    round_num: int,
) -> str:
    prompt = f"""
You are acting as [{agent["name"]}] in a precedent stress-test session.

Your role:
{agent["instructions"]}

Required output format:
{agent["turn_format"]}

Current matter:
{json.dumps(new_matter, indent=2)}

Retrieved internal precedents:
INTERNAL_START
{internal_context}
INTERNAL_END

Retrieved external public context:
EXTERNAL_START
{external_context}
EXTERNAL_END

Prior discussion:
DISCUSSION_START
{prior_discussion if prior_discussion.strip() else "(none yet)"}
DISCUSSION_END

Current round:
{round_num}

Rules:
- Be specific, not generic
- Do not invent facts
- Do not repeat prior points unless directly refining or challenging them
- Start exactly with "[{agent['name']}]:"
- Output only your answer
"""
    out = llm_complete(agent["system"], prompt)
    if not out.startswith(f"[{agent['name']}]"):
        out = f"[{agent['name']}]: {out}"
    return out


def run_debate(
    new_matter: Dict[str, Any],
    internal_context: str,
    external_context: str,
    rounds: int = 1,
) -> str:
    transcript = []
    for round_num in range(1, rounds + 1):
        for agent in AGENTS:
            prior = "\n".join(transcript[-10:]) if transcript else ""
            turn = run_agent_turn(
                agent=agent,
                new_matter=new_matter,
                internal_context=internal_context,
                external_context=external_context,
                prior_discussion=prior,
                round_num=round_num,
            )
            transcript.append(turn)
    return "\n\n".join(transcript)


# =========================================================
# FINAL MEMO
# =========================================================

def build_final_report(
    new_matter: Dict[str, Any],
    internal_context: str,
    external_context: str,
    discussion: str,
) -> str:
    prompt = f"""
You are preparing a final memo for a law firm team using an internal precedent stress-test system.

Current matter:
{json.dumps(new_matter, indent=2)}

Internal precedents:
{internal_context}

External context:
{external_context}

Debate transcript:
{discussion}

Write a concise memo in Markdown with exactly these sections:

# Final Precedent Stress-Test Memo

## 1. Best Internal Analogy
## 2. Where That Analogy Breaks
## 3. External Development That Matters Most
## 4. Reusable Lesson
## 5. What The Team Should Do Now
## 6. Confidence

Requirements:
- Be concrete and comparative
- Explicitly name the best precedent by ID if one exists
- Explicitly state one thing the team should reuse
- Explicitly state one thing the team should not assume
- Under "What The Team Should Do Now", provide exactly 3 numbered actions
- Write like a sharp internal strategy note, not a generic AI summary
- Use law-firm practical language
- Prefer language like: develop, preserve, pressure-test, prepare fallback positions, engage early, refine arguments
"""
    return llm_complete(
        "You write sharp, partner-ready internal legal strategy memos.",
        prompt
    )


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":
    new_matter = {
        "matter_name": "Potential Antitrust Review for Tech Platform Expansion",
        "practice_area": "Antitrust",
        "industry": "Technology",
        "client_type": "Public Company",
        "matter_summary": (
            "A large technology platform is facing growing scrutiny over alleged exclusionary conduct "
            "connected to bundling, platform control, and competitive effects in a core ecosystem. "
            "The client wants to reduce enforcement risk, preserve strategic flexibility, and avoid "
            "structural remedies while maintaining a pro-innovation narrative."
        ),
        "key_issues": [
            "bundling",
            "platform control",
            "market power",
            "regulatory scrutiny",
            "behavioral vs structural remedies",
        ],
    }

    core_matters = generate_core_past_matters()
    synthetic_matters = generate_large_synthetic_dataset(n=40)
    past_matters = core_matters + synthetic_matters

    similar = retrieve_similar_matters(
        new_matter=new_matter,
        past_matters=past_matters,
        top_k=3,
        min_similarity=0.05,
    )
    internal_context = format_internal_precedents(similar)

    queries = generate_tavily_queries(new_matter)
    tavily_packet = tavily_search_packet(queries, max_results=5)
    external_context = format_external_context(tavily_packet, max_items=4)

    discussion = run_debate(
        new_matter=new_matter,
        internal_context=internal_context,
        external_context=external_context,
        rounds=1,
    )

    final_report = build_final_report(
        new_matter=new_matter,
        internal_context=internal_context,
        external_context=external_context,
        discussion=discussion,
    )

    print("\n" + "=" * 100)
    print("CURRENT MATTER")
    print("=" * 100)
    print(json.dumps(new_matter, indent=2))

    print("\n" + "=" * 100)
    print("RETRIEVED INTERNAL PRECEDENTS")
    print("=" * 100)
    print(internal_context)

    print("\n" + "=" * 100)
    print("TAVILY QUERIES")
    print("=" * 100)
    print(json.dumps(queries, indent=2))

    print("\n" + "=" * 100)
    print("TAVILY SOURCES")
    print("=" * 100)
    print(format_external_sources(tavily_packet))

    print("\n" + "=" * 100)
    print("MULTI-AGENT DEBATE")
    print("=" * 100)
    print(discussion)

    print("\n" + "=" * 100)
    print("FINAL PRECEDENT STRESS-TEST MEMO")
    print("=" * 100)
    print(final_report)


CURRENT MATTER
{
  "matter_name": "Potential Antitrust Review for Tech Platform Expansion",
  "practice_area": "Antitrust",
  "industry": "Technology",
  "client_type": "Public Company",
  "matter_summary": "A large technology platform is facing growing scrutiny over alleged exclusionary conduct connected to bundling, platform control, and competitive effects in a core ecosystem. The client wants to reduce enforcement risk, preserve strategic flexibility, and avoid structural remedies while maintaining a pro-innovation narrative.",
  "key_issues": [
    "bundling",
    "platform control",
    "market power",
    "regulatory scrutiny",
    "behavioral vs structural remedies"
  ]
}

RETRIEVED INTERNAL PRECEDENTS
[Internal Precedent 1]
ID: M001
Practice Area: Antitrust
Industry: Technology
Client Type: Public Company
Similarity Score: 0.491
Matter Summary: Technology platform company faced antitrust scrutiny over alleged exclusionary conduct, bundling practices, and market power in a cor